In [1]:
import pandas as pd
from pathlib import Path
import kagglehub

# Download latest version
# ใช้ชุดข้อมูลประกันเดียวกันทุกครั้งเพื่อให้การสำรวจอ้างอิงแหล่งข้อมูลตรงกัน
path = kagglehub.dataset_download("anmolkumar/health-insurance-cross-sell-prediction")

print("Path to dataset files:", path)


Path to dataset files: C:\Users\Lenovo\.cache\kagglehub\datasets\anmolkumar\health-insurance-cross-sell-prediction\versions\1


In [3]:
files = list(Path(path).iterdir())
print(files)

[WindowsPath('C:/Users/Lenovo/.cache/kagglehub/datasets/anmolkumar/health-insurance-cross-sell-prediction/versions/1/sample_submission.csv'), WindowsPath('C:/Users/Lenovo/.cache/kagglehub/datasets/anmolkumar/health-insurance-cross-sell-prediction/versions/1/test.csv'), WindowsPath('C:/Users/Lenovo/.cache/kagglehub/datasets/anmolkumar/health-insurance-cross-sell-prediction/versions/1/train.csv')]


In [4]:
# จากรายการไฟล์ด้านบน files[2] คือ train.csv ซึ่งมี response สำหรับวิเคราะห์การตอบรับข้อเสนอประกัน
df = pd.read_csv(files[2])
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 381109 entries, 0 to 381108
Data columns (total 12 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   id                    381109 non-null  int64  
 1   Gender                381109 non-null  str    
 2   Age                   381109 non-null  int64  
 3   Driving_License       381109 non-null  int64  
 4   Region_Code           381109 non-null  float64
 5   Previously_Insured    381109 non-null  int64  
 6   Vehicle_Age           381109 non-null  str    
 7   Vehicle_Damage        381109 non-null  str    
 8   Annual_Premium        381109 non-null  float64
 9   Policy_Sales_Channel  381109 non-null  float64
 10  Vintage               381109 non-null  int64  
 11  Response              381109 non-null  int64  
dtypes: float64(3), int64(6), str(3)
memory usage: 40.5 MB


In [5]:
# ชื่อคอลัมน์แบบเดียวกันช่วยลดความเสี่ยงอ้างตัวแปรผิดเมื่อเทียบข้อมูล train/test
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

In [6]:
# แถวที่ซ้ำทั้งระเบียนอาจทำให้จำนวนลูกค้าและอัตราตอบรับถูกนับเกินจริง
df.duplicated().sum()

np.int64(0)

In [7]:
# id ที่ซ้ำอาจทำให้ลูกค้าคนเดียวมีน้ำหนักมากเกินไป แม้ค่าคอลัมน์อื่นจะต่างกัน
df["id"].duplicated().sum()

np.int64(0)

In [9]:
df.columns.tolist()

['id',
 'gender',
 'age',
 'driving_license',
 'region_code',
 'previously_insured',
 'vehicle_age',
 'vehicle_damage',
 'annual_premium',
 'policy_sales_channel',
 'vintage',
 'response']

In [10]:
# ตรวจเศษทศนิยมก่อนแปลงรหัสหมวดหมู่ เพื่อไม่ให้รหัสภูมิภาคหรือช่องทางขายถูกตัดค่า
for col in ["region_code", "policy_sales_channel"]:
    print(col, "ค่าที่มีทศนิยม", (df[col] % 1 != 0).sum())

region_code ค่าที่มีทศนิยม 0
policy_sales_channel ค่าที่มีทศนิยม 0


In [14]:
# รหัสภูมิภาคและช่องทางขายเป็นตัวระบุหมวดหมู่ ไม่ใช่ค่าปริมาณต่อเนื่อง; ผลตรวจด้านบนพบว่าไม่มีเศษทศนิยม
for col in ["region_code", "policy_sales_channel"]:
    df[col] = df[col].astype("int64")

In [18]:

print(df[["region_code", "policy_sales_channel"]].dtypes)

region_code             int64
policy_sales_channel    int64
dtype: object


In [22]:
# ตรวจค่าหมวดหมู่เพื่อจับชื่อที่สะกดไม่ตรงกันหรือค่าที่ไม่คาดคิดก่อนใช้แบ่งกลุ่มลูกค้า
for col in ["gender","vehicle_age","vehicle_damage"]:
    print(df[col].value_counts(dropna=False))
    print()

gender
Male      206089
Female    175020
Name: count, dtype: int64

vehicle_age
1-2 Year     200316
< 1 Year     164786
> 2 Years     16007
Name: count, dtype: int64

vehicle_damage
Yes    192413
No     188696
Name: count, dtype: int64



In [23]:
# สัดส่วน response ที่ไม่สมดุลมีผลต่อการตีความอัตราตอบรับและการสร้างแบบจำลอง
for col in ["driving_license","previously_insured","response"]:
    print(df[col].value_counts(dropna=False))
    print()

driving_license
1    380297
0       812
Name: count, dtype: int64

previously_insured
0    206481
1    174628
Name: count, dtype: int64

response
0    334399
1     46710
Name: count, dtype: int64



In [24]:
# ตรวจค่าปลายหางของเบี้ยประกันก่อนกำหนดวิธีจัดการ เพราะเบี้ยสูงอาจเป็นกรมธรรม์จริง
df.describe()

,id,age,driving_license,region_code,previously_insured,annual_premium,policy_sales_channel,vintage,response
count,381109.000000,381109.000000,381109.000000,381109.000000,381109.000000,381109.000000,381109.000000,381109.000000,381109.000000
mean,190555.000000,38.822584,0.997869,26.388807,0.458210,30564.389581,112.034295,154.347397,0.122563
std,110016.836208,15.511611,0.046110,13.229888,0.498251,17213.155057,54.203995,83.671304,0.327936
min,1.000000,20.000000,0.000000,0.000000,0.000000,2630.000000,1.000000,10.000000,0.000000
25%,95278.000000,25.000000,1.000000,15.000000,0.000000,24405.000000,29.000000,82.000000,0.000000
50%,190555.000000,36.000000,1.000000,28.000000,0.000000,31669.000000,133.000000,154.000000,0.000000
75%,285832.000000,49.000000,1.000000,35.000000,1.000000,39400.000000,152.000000,227.000000,0.000000
max,381109.000000,85.000000,1.000000,52.000000,1.000000,540165.000000,163.000000,299.000000,1.000000
